# Environment Setup & Dependencies


In [ ]:
!pip install -q git+https://github.com/huggingface/transformers accelerate qwen-vl-utils[decord] pillow
!pip install -q openai-whisper

# Model Initialization

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

# Library imports

In [ ]:
import io
import json
from base64 import b64decode
import torch
import whisper
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from decord import VideoReader, cpu
from google.colab import files
from google.colab.output import eval_js
from IPython.display import Javascript, display

# Upload and view image

In [ ]:
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

image = Image.open(image_path)

image

# Image-Question Function

In [ ]:
# Load Whisper
whisper_model = whisper.load_model("base")

# Record audio and capture a frame from the webcam
def record_audio_and_capture(seconds=5, silence_threshold=0.02, max_wait=10):
    """Waits for you to actually start speaking (voice detection), THEN records
    for `seconds` and grabs a frame at the end — so pauses before you speak
    don't eat into your question, and the image matches what you were asking about.
    max_wait = how many seconds it'll wait in silence before giving up."""
    js = Javascript("""
    async function recordAudioAndCapture(seconds, silenceThreshold, maxWait){
      if (!window.video) { throw new Error("Call start_webcam() first."); }

      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      // Voice activity detection: wait until you actually start speaking
      const audioContext = new (window.AudioContext || window.webkitAudioContext)();
      const source = audioContext.createMediaStreamSource(stream);
      const analyser = audioContext.createAnalyser();
      analyser.fftSize = 2048;
      source.connect(analyser);
      const dataArray = new Uint8Array(analyser.fftSize);

      function getVolume() {
        analyser.getByteTimeDomainData(dataArray);
        let sumSquares = 0;
        for (const amp of dataArray) {
          const norm = (amp - 128) / 128;
          sumSquares += norm * norm;
        }
        return Math.sqrt(sumSquares / dataArray.length);
      }

      const waitStart = Date.now();
      while (getVolume() < silenceThreshold) {
        if (Date.now() - waitStart > maxWait * 1000) break; // don't wait forever
        await new Promise(r => setTimeout(r, 100));
      }

      // Speech detected (or timed out) -> now record the actual question
      await new Promise(resolve => setTimeout(resolve, seconds * 1000));

      const video = window.video;
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      const frameData = canvas.toDataURL('image/jpeg');

      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      audioContext.close();

      let blob = new Blob(chunks, { type: 'audio/wav' });
      let buffer = await blob.arrayBuffer();
      let binary = '';
      new Uint8Array(buffer).forEach(b => binary += String.fromCharCode(b));
      const audioData = btoa(binary);

      return JSON.stringify({ audio: audioData, frame: frameData });
    }
    """)
    display(js)
    result = eval_js(f"recordAudioAndCapture({seconds}, {silence_threshold}, {max_wait})")
    parsed = json.loads(result)

    audio_bytes = b64decode(parsed["audio"])
    frame_binary = b64decode(parsed["frame"].split(',')[1])
    image = Image.open(io.BytesIO(frame_binary))

    return audio_bytes, image

# Process image, record voice question, and get answer from Qwen
def ask_qwen_multimodal(image, seconds=5):
    # Voice question
    print("Speak your question...")
    audio_bytes = record_audio(seconds)
    filename = "question.wav"

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    # Whisper speech -> text
    result = whisper_model.transcribe(filename)
    voice_question = result["text"].strip()
    print("You said:", voice_question)

    # Optional typed addition
    extra_text = input("Want to add anything? : ")
    no_words = ["no", "nope", "nah", "nothing", "nahi", "nh", "n", "ni", ""]

    if extra_text.strip().lower() in no_words:
        final_question = voice_question
    else:
        final_question = voice_question + " " + extra_text

    print("Final Question:", final_question)

    # Qwen message
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": final_question
                }
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    )
    inputs = inputs.to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100
        )

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]

    answer = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    return answer[0]

# Test Model with an Image

In [ ]:
answer = ask_qwen_multimodal(image)

print("\nAnswer:")
print(answer)

# Video Processing

In [ ]:
# Process video frames, record voice question, and get answer from Qwen
def ask_qwen_video_voice(video_path, seconds=5, num_frames=4):
    # Load video and extract frames evenly spaced throughout the duration
    vr = VideoReader(video_path, ctx=cpu())
    total_frames = len(vr)

    frame_indices = [
        int(i * total_frames / num_frames)
        for i in range(num_frames)
    ]
    frames = vr.get_batch(frame_indices).asnumpy()

    images = [
        Image.fromarray(frame)
        for frame in frames
    ]

    # Record voice question and transcribe using Whisper
    print("Speak your question...")
    audio_bytes = record_audio(seconds)
    filename = "video_question.wav"

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    result = whisper_model.transcribe(filename)
    voice_question = result["text"].strip()
    print("You said:", voice_question)

    # Allow user to add optional text to the voice prompt
    extra_text = input("Do you want to add anything?: ")

    no_words = [
        "no", "nope", "nah", "nothing", "nahi", "nh", "n", "ni", ""
    ]

    if extra_text.lower().strip() in no_words:
        final_question = voice_question
    else:
        final_question = voice_question + " " + extra_text

    print("Final Question:", final_question)

    # Format messages and run the Qwen model
    messages = [
        {
            "role": "user",
            "content": []
        }
    ]

    # Add video frames to the prompt
    for img in images:
        messages[0]["content"].append(
            {
                "type": "image",
                "image": img
            }
        )

    # Add text question to the prompt
    messages[0]["content"].append(
        {
            "type": "text",
            "text": final_question
        }
    )

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=images,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50
        )

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]

    answer = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    return answer[0]

# Test Model with a Video

In [ ]:
# Set the path to your video file
video_path = "/content/Qwent2.mp4"  # Add your video path here

# Run the multimodal video query
answer = ask_qwen_video_voice(
    video_path=video_path,
    seconds=5,
    num_frames=4
)

print("\nAnswer:")
print(answer)

# Real-Time Webcam Integration


In [ ]:
whisper_model = whisper.load_model("base")

# Persistent webcam setup functions
def start_webcam():
    """Turns the camera on ONCE. Keeps running until stop_webcam() is called."""
    js = Javascript('''
    async function startWebcam() {
      if (window.stream) { return "already running"; }

      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.transform = 'scaleX(-1)';   // mirror the preview only
      video.setAttribute('playsinline', '');

      const stream = await navigator.mediaDevices.getUserMedia({ video: true });

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;

      await new Promise((resolve) => { video.onloadedmetadata = resolve; });
      await video.play();

      window.stream = stream;
      window.video = video;
      window.videoDiv = div;
      return "started";
    }
    ''')
    display(js)
    print("Webcam:", eval_js('startWebcam()'))

def capture_frame():
    """Grabs ONE frame from the already-running stream. Does NOT stop the camera.
    NOT mirrored — keeps the image true to reality for accurate analysis."""
    js = Javascript('''
    async function captureFrame() {
      if (!window.video) { throw new Error("Call start_webcam() first."); }
      const video = window.video;
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg');
    }
    ''')
    display(js)
    data = eval_js('captureFrame()')
    binary = b64decode(data.split(',')[1])
    return Image.open(io.BytesIO(binary))

def stop_webcam():
    """Turns the camera off. Call this only when you're completely done."""
    js = Javascript('''
    function stopWebcam() {
      if (window.stream) {
        window.stream.getVideoTracks()[0].stop();
        window.videoDiv.remove();
        window.stream = null;
        window.video = null;
        return "stopped";
      }
      return "was not running";
    }
    ''')
    display(js)
    print("Webcam:", eval_js('stopWebcam()'))

# Record audio and capture a synchronized frame
def record_audio_and_capture(seconds=5, silence_threshold=0.02, max_wait=10):
    """Waits for you to actually start speaking (voice detection), THEN records
    for `seconds` and grabs a frame at the end — so pauses before you speak
    don't eat into your question, and the image matches what you were asking about.
    max_wait = how many seconds it'll wait in silence before giving up."""
    js = Javascript("""
    async function recordAudioAndCapture(seconds, silenceThreshold, maxWait){
      if (!window.video) { throw new Error("Call start_webcam() first."); }

      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      // Voice activity detection: wait until you actually start speaking
      const audioContext = new (window.AudioContext || window.webkitAudioContext)();
      const source = audioContext.createMediaStreamSource(stream);
      const analyser = audioContext.createAnalyser();
      analyser.fftSize = 2048;
      source.connect(analyser);
      const dataArray = new Uint8Array(analyser.fftSize);

      function getVolume() {
        analyser.getByteTimeDomainData(dataArray);
        let sumSquares = 0;
        for (const amp of dataArray) {
          const norm = (amp - 128) / 128;
          sumSquares += norm * norm;
        }
        return Math.sqrt(sumSquares / dataArray.length);
      }

      const waitStart = Date.now();
      while (getVolume() < silenceThreshold) {
        if (Date.now() - waitStart > maxWait * 1000) break; // don't wait forever
        await new Promise(r => setTimeout(r, 100));
      }

      // Speech detected (or timed out) -> now record the actual question
      await new Promise(resolve => setTimeout(resolve, seconds * 1000));

      const video = window.video;
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      const frameData = canvas.toDataURL('image/jpeg');

      recorder.stop();
      await new Promise(resolve => recorder.onstop = resolve);
      stream.getTracks().forEach(track => track.stop());
      audioContext.close();

      let blob = new Blob(chunks, { type: 'audio/wav' });
      let buffer = await blob.arrayBuffer();
      let binary = '';
      new Uint8Array(buffer).forEach(b => binary += String.fromCharCode(b));
      const audioData = btoa(binary);

      return JSON.stringify({ audio: audioData, frame: frameData });
    }
    """)
    display(js)
    result = eval_js(f"recordAudioAndCapture({seconds}, {silence_threshold}, {max_wait})")
    parsed = json.loads(result)

    audio_bytes = b64decode(parsed["audio"])
    frame_binary = b64decode(parsed["frame"].split(',')[1])
    image = Image.open(io.BytesIO(frame_binary))

    return audio_bytes, image

# Main loop to ask Qwen questions using the webcam
def ask_qwen_frame(seconds=5):
    """Asks ONCE whether you want to type or speak your questions, then keeps
    looping in that mode: capture a matching frame, get your question, ask Qwen,
    print the answer, and listen/read again. Call start_webcam() once before
    your first call to this. Type 'stop' as your question whenever you're done."""

    mode = input("Do you want to type your questions, or speak them? (type/speak): ").strip().lower()
    speak_mode = mode.startswith("s")  # anything starting with 's' -> speak, else -> type

    while True:
        if speak_mode:
            print("Point at what you're asking about and speak your question...")
            audio_bytes, image = record_audio_and_capture(seconds)

            filename = "webcam_question.wav"
            with open(filename, "wb") as f:
                f.write(audio_bytes)

            result = whisper_model.transcribe(filename)
            final_question = result["text"].strip()
            print("You said:", final_question)
        else:
            image = capture_frame()
            final_question = input("Type your question ('stop' to end): ").strip()

        stop_words = [
           "stop!", "Stop!", "stop", "end", "exit", "quit"
        ]

        if final_question.strip().lower() in stop_words:
            print("Stopping.")
            break

        print("Final Question:", final_question)

        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": final_question},
        ]}]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(**inputs, max_new_tokens=200)

        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        print("Answer:", answer)
        print("-" * 40)

# Run the Live Webcam Assistant

In [ ]:
# Turn the camera on once
start_webcam()

# Ask for input mode, then loop questions until you say or type 'stop'
ask_qwen_frame(seconds=5)

# Safely turn the camera off when the loop ends
stop_webcam()